In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class MNIST(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MNIST, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        # Reshape the input if it's not already flattened
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        return self.net(x)

# Load MNIST dataset
train_dataset = datasets.MNIST(
    root='data',
    train=True,
    transform=transforms.ToTensor(),
    download=True
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,  # Number of samples per batch
    shuffle=True    # Shuffle data every epoch
)

# Initialize model, loss function, and optimizer
model = MNIST(28*28, 128, 10)  # MNIST images are 28x28 pixels
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 12
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, targets in train_loader:
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Print statistics
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}')

print('Training complete!')

Epoch 1/12, Loss: 0.3373
Epoch 2/12, Loss: 0.1528
Epoch 3/12, Loss: 0.1074
Epoch 4/12, Loss: 0.0821
Epoch 5/12, Loss: 0.0644
Epoch 6/12, Loss: 0.0530
Epoch 7/12, Loss: 0.0429
Epoch 8/12, Loss: 0.0345
Epoch 9/12, Loss: 0.0289
Epoch 10/12, Loss: 0.0238
Epoch 11/12, Loss: 0.0197
Epoch 12/12, Loss: 0.0164
Training complete!


In [9]:
def evaluate(model, test_loader):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():  # Disable gradient calculation to save memory during inference
        for inputs, targets in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
            total += targets.size(0)  # Total number of images
            correct += (predicted == targets).sum().item()  # Count correct predictions

    accuracy = 100 * correct / total
    return accuracy
test_loader = DataLoader(
    dataset=train_dataset,  # Use test dataset here
    batch_size=64,  # Reasonable batch size
    shuffle=False  # Don't shuffle test data for evaluation
)

# Evaluate the model on the test set
accuracy = evaluate(model, test_loader)
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 99.66%
